## MCP

- A standard protocol that allows AI models to communicate with external tools and data sources in a structured, interoperable way.

- Break it down word by word:
- Model : The LLM (Claude, GPT, etc.)
- Context : Information the LLM can see and use
- Protocol : A set of rules for communication (like HTTP is a protocol for the web)

#### So literally: "Rules for how an AI model exchanges context with the outside world."

## How MCP Works

MCP acts like a **USB-C port for AI** — a universal, standardized way to plug any AI model into any external tool or data source without custom integrations.

---

## Core Components

### 1. MCP Host
- The application the user interacts with (e.g., Claude Desktop, Cursor, VS Code with AI extensions)
- Manages the overall session and coordinates between the user and the AI model

### 2. MCP Client
- Lives inside the Host
- Converts user requests into structured protocol messages
- Maintains a **1:1 relationship** with a single MCP Server
- Multiple clients can exist within one Host

### 3. MCP Server
- External service or process that exposes capabilities to the LLM
- Connects to databases, APIs, file systems, web services, etc.
- Responds to requests from the MCP Client

---

## Core Primitives (What a Server Can Expose)

| Primitive | Description | Example |
|-----------|-------------|---------|
| **Tools** | Actions the AI can trigger | Run a query, send an email, search the web |
| **Resources** | Structured data the AI can read | Files, database records, documentation |
| **Prompts** | Predefined instruction templates | "Summarize this repo", "Review this PR" |

---

## Communication Protocol

MCP uses **JSON-RPC 2.0** for all client-server communication.

### Transport Mechanisms
- **Stdio (Standard I/O)** — for local processes on the same machine; fast and simple
- **Streamable HTTP** — for remote servers; uses HTTP POST + Server-Sent Events (SSE); supports auth

### Session Lifecycle
```
1. initialize    →  Client connects and sends capabilities
2. negotiate     →  Server responds with its own capabilities
3. ready         →  Both sides confirm, session is active
4. request/respond  →  Normal tool/resource/prompt exchanges
5. shutdown      →  Session cleanly terminated
```

---

## End-to-End Flow (Example)

```
User: "What files changed in the last commit?"

[Host] → receives user message
  ↓
[MCP Client] → formats it as a JSON-RPC request
  ↓
[MCP Server (Git Tool)] → runs `git log -1 --name-only`
  ↓
[MCP Client] → receives result
  ↓
[LLM] → processes result and generates a response
  ↓
User sees: "The following files changed: main.py, utils.py..."
```

---

## Why MCP Matters

- **No more custom glue code** — one protocol works across all AI models and tools
- **Interoperability** — any MCP-compatible host can connect to any MCP-compatible server
- **Open standard** — donated to the Linux Foundation (Agentic AI Foundation) in December 2025
- **Scalable** — easily add new tools/resources without changing the AI model

---

> **In short:** MCP standardizes how AI models get context from the outside world — making AI integrations modular, reusable, and model-agnostic.

## Simple MCP Server — Addition Tool

A minimal MCP server that exposes one tool: `add(a, b)`.  
Run it with `python addition_server.py` — it communicates over **stdio** (standard input/output).

In [1]:
# addition_server.py  —  Simple MCP server with an addition tool

server_code = '''
from mcp.server.fastmcp import FastMCP

# Create the MCP server and give it a name
mcp = FastMCP("Addition Server")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b


if __name__ == "__main__":
    mcp.run()   # runs over stdio by default
'''

# Write the server to a file so it can be run directly
with open("addition_server.py", "w") as f:
    f.write(server_code.strip())

print("addition_server.py created successfully!")

addition_server.py created successfully!


### Test the tool directly (in-process)

No need to spin up the server just to verify the logic — call the function directly.

In [2]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Addition Server")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

# Test the tool directly
print(add(3, 5))      # → 8.0
print(add(10.5, 4.5)) # → 15.0
print(add(-2, 7))     # → 5.0

8
15.0
5


### How to connect a client to this server

```python
# client.py — connect to the addition server via stdio
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    server_params = StdioServerParameters(
        command="python",
        args=["addition_server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Call the add tool
            result = await session.call_tool("add", {"a": 10, "b": 25})
            print("Result:", result.content[0].text)  # → 35.0

asyncio.run(main())
```

> Run `python client.py` in the terminal after the server file is created.

---

## MCP Calculator — with User Input

A full calculator MCP server exposing **add, subtract, multiply, divide** as tools.  
The client takes values from the user via `input()` and calls the matching tool on the server.

### Step 1 — Calculator Server (`calculator_server.py`)

In [3]:
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Calculator Server")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@mcp.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises error if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b


if __name__ == "__main__":
    mcp.run()
'''

with open("calculator_server.py", "w") as f:
    f.write(server_code.strip())

print("calculator_server.py created!")

calculator_server.py created!


### Step 2 — Calculator Client (`calculator_client.py`)

Takes user input, launches the server as a subprocess over stdio, and calls the right tool.

In [ ]:
client_code = '''
import asyncio
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "calculator_server.py")

OPERATIONS = {
    "1": ("add",      "+"),
    "2": ("subtract", "-"),
    "3": ("multiply", "*"),
    "4": ("divide",   "/"),
}

async def run_calculator():
    server_params = StdioServerParameters(
        command="python",
        args=[SERVER_PATH]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            print("\\n=============================")
            print("   MCP Calculator")
            print("=============================")

            while True:
                print("\\nSelect operation:")
                for key, (name, symbol) in OPERATIONS.items():
                    print(f"  {key}. {name.capitalize()} ({symbol})")
                print("  5. Exit")

                choice = input("\\nEnter choice (1-5): ").strip()

                if choice == "5":
                    print("Goodbye!")
                    break

                if choice not in OPERATIONS:
                    print("Invalid choice. Try again.")
                    continue

                try:
                    a = float(input("Enter first number : "))
                    b = float(input("Enter second number: "))
                except ValueError:
                    print("Invalid number. Please enter digits only.")
                    continue

                tool_name, symbol = OPERATIONS[choice]

                try:
                    result = await session.call_tool(tool_name, {"a": a, "b": b})
                    answer = result.content[0].text
                    print(f"\\nResult: {a} {symbol} {b} = {answer}")
                except Exception as e:
                    print(f"Error: {e}")

asyncio.run(run_calculator())
'''

with open("calculator_client.py", "w") as f:
    f.write(client_code.strip())

print("calculator_client.py created!")

### How to Run

1. **Execute both cells above** to generate the two `.py` files.
2. Open a terminal and run:
   ```bash
   python calculator_client.py
   ```
3. You'll see an interactive menu like:

```
=============================
   MCP Calculator
=============================

Select operation:
  1. +
  2. -
  3. *
  4. /
  5. Exit

Enter choice (1-5): 3
Enter first number : 6
Enter second number: 7

Result: 6.0 * 7.0 = 42.0
```

> The client **automatically starts the server** as a subprocess — you only need to run `calculator_client.py`.

### What's happening under the hood

```
calculator_client.py
  │
  ├─ spawns ──► calculator_server.py  (subprocess over stdio)
  │                  │
  │             FastMCP receives JSON-RPC call
  │             runs add / subtract / multiply / divide
  │             returns result over stdout
  │
  └─ prints result to user
```